# CHAPTER 6
# Data Wrangling

In many applications, data may be spread across a number of files or databases or be arranged in a form that is not easy to analyze. This chapter focuses on tools to help combine, join, and rearrange data. 

## 6.1 Hierarchical Indexing 

Hierarchical indexing is an important feature of pandas that enables you to have multiple (two or more) index levels on an axis. Somewhat abstractly, it provides a way for you to work with higher dimensional data in a lower dimensional form. Let’s start with a simple example; create a Series with a list of lists (or arrays) as the index:

In [7]:
import pandas as pd
import numpy as np

In [8]:
data = pd.DataFrame(np.random.randn(9),index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                                           [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data 

0
a 1  0.118775
  2  0.872854
  3 -1.282043
b 1  0.296106
  3 -1.479621
c 1 -0.984082
  2  1.252534
d 2  0.326968
  3  0.854566

What you’re seeing is a prettified view of a Series with a MultiIndex as its index. The “gaps” in the index display mean “use the label directly above”:

In [9]:
data.index 

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

With a hierarchically indexed object, so-called partial indexing is possible, enabling you to concisely select subsets of the data:

In [10]:
data['b':'c'] 

0
b 1  0.296106
  3 -1.479621
c 1 -0.984082
  2  1.252534

Selection is even possible from an “inner” level:

In [11]:
data

0
a 1  0.118775
  2  0.872854
  3 -1.282043
b 1  0.296106
  3 -1.479621
c 1 -0.984082
  2  1.252534
d 2  0.326968
  3  0.854566

In [12]:
data.loc[:, 2] # iloc is for index and not label(in this case)

KeyError: 2

In [13]:
data[:, 2] # same as without using a .loc

KeyError: (slice(None, None, None), 2)

Hierarchical indexing plays an important role in reshaping data and group-based operations like forming a pivot table. For example, you could rearrange the data into a DataFrame using its **unstack** method:

In [ ]:
data.unstack() 

The inverse operation of **unstack** is **stack**:

In [ ]:
data.unstack().stack() 

With a DataFrame, either axis can have a hierarchical index:

In [ ]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)), index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]], 
                     columns=[['Ohio', 'Ohio', 'Colorado'],['Green', 'Red', 'Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

The hierarchical levels can have names (as strings or any Python objects). If so, these will show up in the console output:

In [ ]:
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

With partial column indexing you can similarly select groups of columns:

In [ ]:
frame['Ohio'] # get all except for Colorado

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A MultiIndex can be created by itself and then reused; the columns in the preceding DataFrame with level names could be created like this:

In [ ]:
pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                       names=['state', 'color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

In [ ]:
result = [['A', np.nan], ['A', np.nan], [np.nan,'A-']]

data1 = pd.DataFrame(result, index=[['SECP1234', 'SECP5678', 'SECP0123']],
                     columns=[['SEM 1', 'SEM 2']])

data1

,SEM 1,SEM 2
SECP1234,A,NaN
SECP5678,A,NaN
SECP0123,NaN,A-


In [ ]:
#data1.index.names = 
#pd.MultiIndex.from_arrays([['SEM 1', 'SEM 1', 'SEM 2'], ['SECP1234', 'SECP5678'], ['SECP0123']],
                          #names=['Result', 'SEM'])

# Stack the DataFrame to reshape it
# = df.stack().reset_index()
#df_stacked.columns = ['Student', 'Semester', 'Grade']

# Set MultiIndex: Semester as outer, Student as inner
#df_multi = df_stacked.set_index(['Semester', 'Student'])

#print("\nRestructured DataFrame with MultiIndex:")
#print(df_multi)


### 6.1.1 Reordering and Sorting Levels 

At times you will need to rearrange the order of the levels on an axis or sort the data by the values in one specific level. The **swaplevel** takes two level numbers or names and returns a new object with the levels interchanged (but the data is otherwise unaltered):

In [ ]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [ ]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

**sort_index**, on the other hand, sorts the data using only the values in a single level. When swapping levels, it’s not uncommon to also use **sort_index** so that the result is lexicographically sorted by the indicated level:

In [ ]:
frame.sort_index(level=1) 

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [ ]:
frame.swaplevel(0, 1).sort_index(level=0) 

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

### 6.1.2 Summary Statistics by Level 

Many descriptive and summary statistics on DataFrame and Series have a level option in which you can specify the level you want to aggregate by on a particular axis. Consider the above DataFrame; we can aggregate by level on either the rows or columns like so:

In [ ]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [ ]:
frame.sum(level='key2') 

TypeError: sum() got an unexpected keyword argument 'level'

In [ ]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [ ]:
frame.groupby(level='color', axis=1) 

C:\Users\hp\AppData\Local\Temp\ipykernel_12292\2357374610.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  frame.groupby(level='color', axis=1)


### 6.1.3 Indexing with a DataFrame’s columns 

It’s not unusual to want to use one or more columns from a DataFrame as the row index; alternatively, you may wish to move the row index into the DataFrame’s columns. Here’s an example DataFrame:

**** Breakdown: range(7,0,-1)
- 7 → Start (the first number in the sequence).
- 0 → Stop (the sequence stops before reaching this number).
- -1 → Step (the interval at which numbers decrease).


In [ ]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1), 'c': ['one', 'one', 'one', 'two', 'two','two', 'two'], 
                      'd': [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame’s **set_index** function will create a new DataFrame using one or more of its columns as the index:

In [ ]:
frame

In [ ]:
frame2 = frame.set_index(['c', 'd'])
frame2

By default the columns are removed from the DataFrame, though you can leave them in:

In [ ]:
frame.set_index(['c', 'd'], drop=False) 

**reset_index**, on the other hand, does the opposite of **set_index**; the hierarchical index levels are moved into the columns:

In [ ]:
frame2.reset_index() 

## 6.2 Combining and Merging Datasets 

Data contained in pandas objects can be combined together in a number of ways:

• **pandas.merge** connects rows in DataFrames based on one or more keys. This will be familiar to users of SQL or other relational databases, as it implements database *join* operations. 

• **pandas.concat** concatenates or “stacks” together objects along an axis. 

• The **combine_first** instance method enables splicing together overlapping data to fill in missing values in one object with values from another. 

### 6.2.1 Database-Style DataFrame Joins 

*Merge* or *join* operations combine datasets by linking rows using one or more *keys*. These operations are central to relational databases (e.g., SQL-based). The **merge** function in pandas is the main entry point for using these algorithms on data. Let’s start with a simple example:

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], 'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],'data2': range(3)})
df1 

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [ ]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


This is an example of a *many-to-one* join; the data in *df1* has multiple rows labeled *a* and *b*, whereas *df2* has only one row for each value in the key column. Calling **merge** with these objects we obtain:

In [ ]:
pd.merge(df2, df1) 

,key,data2,data1
0,a,0,2
1,a,0,4
2,a,0,5
3,b,1,0
4,b,1,1
5,b,1,6


In [ ]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Note that we didn’t specify which column to join on. If that information is not specified, **merge** uses the overlapping column names as the keys. It’s a good practice to specify explicitly, though:

In [ ]:
pd.merge?

Signature:
pd.merge(
    left: 'DataFrame | Series',
    right: 'DataFrame | Series',
    how: 'MergeHow' = 'inner',
    on: 'IndexLabel | AnyArrayLike | None' = None,
    left_on: 'IndexLabel | AnyArrayLike | None' = None,
    right_on: 'IndexLabel | AnyArrayLike | None' = None,
    left_index: 'bool' = False,
    right_index: 'bool' = False,
    sort: 'bool' = False,
    suffixes: 'Suffixes' = ('_x', '_y'),
    copy: 'bool | None' = None,
    indicator: 'str | bool' = False,
    validate: 'str | None' = None,
) -> 'DataFrame'
Docstring:
Merge DataFrame or named Series objects with a database-style join.

A named Series object is treated as a DataFrame with a single named column.

The join is done on columns or indexes. If joining columns on
columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
on indexes or indexes on a column or columns, the index will be passed on.
When performing a cross merge, no column specifications to merge on are
allowed.

.. warning:

In [ ]:
pd.merge(df1, df2, on='key') 

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


If the column names are different in each object, you can specify them separately:

In [ ]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],'data1': range(7)})
print(df3)
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'], 'data2': range(3)})
print(df4)
pd.merge(df3, df4, left_on='lkey', right_on='rkey') 

  lkey  data1
0    b      0
1    b      1
2    a      2
3    c      3
4    a      4
5    a      5
6    b      6
  rkey  data2
0    a      0
1    b      1
2    d      2


,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


You may notice that the *'c'* and *'d'* values and associated data are missing from the result. By default **merge** does an **'inner'** join; the keys in the result are the intersection, or the common set found in both tables. Other possible options are **'left'**, **'right'**, and **'outer'**. The outer join takes the union of the keys, combining the effect of applying both left and right joins:

In [ ]:
pd.merge(df1, df2, how='outer') # take all the key and match by default

,key,data1,data2
0,a,2.0,0.0
1,a,4.0,0.0
2,a,5.0,0.0
3,b,0.0,1.0
4,b,1.0,1.0
5,b,6.0,1.0
6,c,3.0,NaN
7,d,NaN,2.0


In [ ]:
pd.merge(df1, df2, how='left') # take all from left(df1) and df2 will find a match

,key,data1,data2
0,b,0,1.0
1,b,1,1.0
2,a,2,0.0
3,c,3,NaN
4,a,4,0.0
5,a,5,0.0
6,b,6,1.0


In [ ]:
pd.merge(df1, df2, how='right') # same as above but df2

,key,data1,data2
0,a,2.0,0
1,a,4.0,0
2,a,5.0,0
3,b,0.0,1
4,b,1.0,1
5,b,6.0,1
6,d,NaN,2


In [ ]:
pd.merge(df1, df2, how='inner') # only a matching one

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


See Table 6-1 for a summary of the options for *how*.

<br>
<center>Table 6.1: Different join types with how argument</center>
<img src="Table6.1.jpg">

*Many-to-many* merges have well-defined, though not necessarily intuitive, behavior. Here’s an example:

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'], 'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'], 'data2': range(5)})
df1 

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [ ]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [ ]:
pd.merge(df1, df2, on='key', how='left') 

,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


*Many-to-many* joins form the Cartesian product of the rows. Since there were three *'b'* rows in the left DataFrame and two in the right one, there are six *'b'* rows in the result. The join method only affects the distinct key values appearing in the result:

In [ ]:
pd.merge(df1, df2, how='inner') 

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


To merge with multiple keys, pass a list of column names:

In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'], 'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})

In [ ]:
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [ ]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [ ]:
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3.0,6.0
1,bar,two,NaN,7.0
2,foo,one,1.0,4.0
3,foo,one,1.0,5.0
4,foo,two,2.0,NaN


A last issue to consider in merge operations is the treatment of overlapping column names. While you can address the overlap manually (see the earlier section on renaming axis labels), merge has a **suffixes** option for specifying strings to append to overlapping names in the left and right DataFrame objects:

In [ ]:
pd.merge(left, right, on='key1') 

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


In [ ]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right')) 

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


See Table 6-2 for an argument reference on **merge**. Joining using the DataFrame’s row index is the subject of the next section.

<br>
<center>Table 6.2: merge function arguments</center>
<img src="Table6.2.jpg">

### 6.2.2 Merging on Index 

In some cases, the merge key(s) in a DataFrame will be found in its index. In this case, you can pass *left_index=True* or *right_index=True* (or both) to indicate that the index should be used as the merge key:

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'], 'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])
left1 

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [ ]:
right1

,group_val
a,3.5
b,7.0


In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


Since the default merge method is to intersect the join keys, you can instead form the union of them with an outer join:

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer') 

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


### 6.2.3 Concatenating Along an Axis 

Another kind of data combination operation is referred to interchangeably as concatenation, binding, or stacking. NumPy’s **concatenate** function can do this with NumPy arrays:

In [ ]:
arr = np.arange(12).reshape((3, 4))
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [ ]:
np.concatenate([arr, arr], axis=1) 

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In the context of pandas objects such as Series and DataFrame, having labeled axes enable you to further generalize array concatenation. In particular, you have a number of additional things to think about: 

• If the objects are indexed differently on the other axes, should we combine the distinct elements in these axes or use only the shared values (the intersection)? 

• Do the concatenated chunks of data need to be identifiable in the resulting object? 

• Does the “concatenation axis” contain data that needs to be preserved? In many cases, the default integer labels in a DataFrame are best discarded during concatenation.

The **concat** function in pandas provides a consistent way to address each of these concerns. We'll give a number of examples to illustrate how it works. Suppose we have three Series with no index overlap:

In [ ]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g']) 

In [ ]:
print(s1)
print()
print(s2)
print()
print(s3)
print()

a    0
b    1
dtype: int64

c    2
d    3
e    4
dtype: int64

f    5
g    6
dtype: int64



Calling **concat** with these objects in a list glues together the values and indexes:

In [ ]:
pd.concat([s1, s2, s3]) 

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: int64

By default **concat** works along axis=0, producing another Series. If you pass axis=1, the result will instead be a DataFrame (axis=1 is the columns):

In [ ]:
pd.concat([s1, s2, s3], axis=1) 

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In this case there is no overlap on the other axis, which as you can see is the sorted union (the **'outer'** join) of the indexes. You can instead intersect them by passing *join='inner'*:

In [ ]:
s4 = pd.concat([s1, s3])
s4

a    0
b    1
f    5
g    6
dtype: int64

In [ ]:
s1

a    0
b    1
dtype: int64

In [ ]:
pd.concat([s1, s4], axis=1)

,0,1
a,0.0,0
b,1.0,1
f,NaN,5
g,NaN,6


In [ ]:
pd.concat([s1, s4])

a    0
b    1
a    0
b    1
f    5
g    6
dtype: int64

In [ ]:
pd.concat([s1, s4], axis=1, join='inner') 

,0,1
a,0,0
b,1,1


In this last example, the *'f'* and *'g'* labels disappeared because of the *join='inner'* option. 

You can even specify the axes to be used on the other axes with **join_axes**:

In [ ]:
pd.concat?

Signature:
pd.concat(
    objs: 'Iterable[Series | DataFrame] | Mapping[HashableT, Series | DataFrame]',
    *,
    axis: 'Axis' = 0,
    join: 'str' = 'outer',
    ignore_index: 'bool' = False,
    keys: 'Iterable[Hashable] | None' = None,
    levels=None,
    names: 'list[HashableT] | None' = None,
    verify_integrity: 'bool' = False,
    sort: 'bool' = False,
    copy: 'bool | None' = None,
) -> 'DataFrame | Series'
Docstring:
Concatenate pandas objects along a particular axis.

Allows optional set logic along the other axes.

Can also add a layer of hierarchical indexing on the concatenation axis,
which may be useful if the labels are the same (or overlapping) on
the passed axis number.

Parameters
----------
objs : a sequence or mapping of Series or DataFrame objects
    If a mapping is passed, the sorted keys will be used as the `keys`
    argument, unless it is passed, in which case the values will be
    selected (see below). Any None objects will be dropped silently unless
  

In [ ]:
desired_index=['a', 'c', 'b', 'e']
#pd.concat([s1, s4], axis=1, join_axes=[['a', 'c', 'b', 'e']])
pd.concat([s1.reindex(desired_index), s4.reindex(desired_index)], axis=1)

,0,1
a,0.0,0.0
c,NaN,NaN
b,1.0,1.0
e,NaN,NaN


A potential issue is that the concatenated pieces are not identifiable in the result. Suppose instead you wanted to create a hierarchical index on the concatenation axis. To do this, use the **keys** argument:

In [ ]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: int64

In [ ]:
result.unstack()

,a,b,f,g
one,0.0,1.0,NaN,NaN
two,0.0,1.0,NaN,NaN
three,NaN,NaN,5.0,6.0


In the case of combining Series along *axis=1*, the keys become the DataFrame column headers:

In [ ]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three']) 

,one,two,three
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


The same logic extends to DataFrame objects:

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],columns=['three', 'four'])
df1 

,one,two
a,0,1
b,2,3
c,4,5


In [ ]:
df2

,three,four
a,5,6
c,7,8


In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

If you pass a dict of objects instead of a list, the dict’s keys will be used for the keys option:

In [ ]:
pd.concat({'level1': df1, 'level2': df2}, axis=1) 

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

See Table 6-3 for arguments reference on **concat**. 

<br>
<center>Table 6.3: concat function arguments</center>
<img src="Table6.3.jpg">

In [ ]:
pd.concat?